<a href="https://colab.research.google.com/github/samyuktha07-prog/Emotion-Switch-Detector/blob/main/emotion_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


!pip -q install gradio pandas numpy matplotlib librosa soundfile

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import librosa
import re
from datetime import datetime


EMOTIONS = {

    "😊 Happy": {
        "words": {
            "happy", "joy", "joyful", "excited", "great",
            "amazing", "wonderful", "love", "loved",
            "fun", "smile", "smiling", "success",
            "successful", "celebrate", "celebration",
            "awesome", "good", "best", "glad", "pleased"
        }
    },

    "😢 Sad": {
        "words": {
            "sad", "cry", "crying", "tears", "hurt",
            "pain", "lonely", "alone", "miss", "missing",
            "broken", "upset", "disappointed", "loss",
            "lost", "depressed", "unhappy", "failure",
            "failed", "sorry"
        }
    },

    "😡 Angry": {
        "words": {
            "angry", "anger", "hate", "hated", "annoyed",
            "annoying", "furious", "mad", "rage", "rude",
            "stupid", "idiot", "worst", "unfair",
            "frustrated", "frustration", "irritated",
            "irritating"
        }
    },

    "😨 Fear": {
        "words": {
            "afraid", "fear", "scared", "scary", "worried",
            "worry", "danger", "dangerous", "nervous",
            "anxious", "anxiety", "terrified", "terror",
            "panic", "panicked", "risk", "threat"
        }
    },

    "😲 Surprise": {
        "words": {
            "surprised", "surprise", "shocked", "shock",
            "wow", "unexpected", "suddenly", "unbelievable",
            "really", "seriously", "omg", "incredible"
        }
    },

    "😍 Love": {
        "words": {
            "love", "lovely", "romantic", "heart",
            "care", "caring", "beautiful", "dear",
            "affection", "kiss", "crush", "together"
        }
    },

    "😐 Neutral": {
        "words": {
            "today", "college", "class", "study", "project",
            "computer", "work", "school", "meeting",
            "assignment", "book", "room"
        }
    }
}


STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were",
    "to", "of", "and", "or", "in", "on", "for",
    "with", "this", "that", "it", "i", "you",
    "he", "she", "we", "they", "my", "your",
    "our", "their", "me", "be", "am", "have",
    "has", "had", "do", "did", "does"
}


analysis_history = []


def split_sentences(text):

    text = text.strip()

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


def tokenize(sentence):

    return re.findall(
        r"\b[a-zA-Z']+\b",
        sentence.lower()
    )


def detect_emotion(sentence):

    words = tokenize(sentence)

    scores = {}

    for emotion, data in EMOTIONS.items():

        score = 0

        for word in words:

            if word in data["words"]:
                score += 1

        scores[emotion] = score


    exclamation_count = sentence.count("!")

    question_count = sentence.count("?")

    punctuation_bonus = min(
        exclamation_count * 0.5 +
        question_count * 0.2,
        2
    )


    capital_words = re.findall(
        r"\b[A-Z]{2,}\b",
        sentence
    )

    capitalization_bonus = min(
        len(capital_words) * 0.4,
        2
    )

    if max(scores.values()) > 0:

        strongest = max(
            scores,
            key=scores.get
        )

        scores[strongest] += (
            punctuation_bonus +
            capitalization_bonus
        )


    best_emotion = max(
        scores,
        key=scores.get
    )

    best_score = scores[best_emotion]

    if best_score == 0:

        best_emotion = "😐 Neutral"
        best_score = 0.5

    intensity = min(
        100,
        int(
            best_score * 25 +
            len(words) * 0.5
        )
    )

    if intensity < 25:
        intensity_level = "Low"
    elif intensity < 55:
        intensity_level = "Medium"
    elif intensity < 80:
        intensity_level = "High"
    else:
        intensity_level = "Extreme"

    return (
        best_emotion,
        intensity,
        intensity_level,
        scores
    )


def detect_switches(results):

    switches = []

    for i in range(1, len(results)):

        previous = results[i - 1]["emotion"]

        current = results[i]["emotion"]

        if previous != current:

            switches.append({
                "from": previous,
                "to": current,
                "sentence": i + 1
            })

    return switches


def analyze_text(text):

    if not text or not text.strip():

        empty_df = pd.DataFrame(
            columns=[
                "Sentence",
                "Emotion",
                "Intensity",
                "Level"
            ]
        )

        return (
            "0",
            "—",
            "—",
            "0",
            "Enter some text first.",
            empty_df,
            None
        )

    sentences = split_sentences(text)

    results = []

    for index, sentence in enumerate(sentences):

        emotion, intensity, level, scores = detect_emotion(
            sentence
        )

        results.append({
            "sentence_no": index + 1,
            "sentence": sentence,
            "emotion": emotion,
            "intensity": intensity,
            "level": level,
            "scores": scores
        })


    switches = detect_switches(results)

    switch_count = len(switches)


    emotion_counts = {}

    for result in results:

        emotion = result["emotion"]

        emotion_counts[emotion] = (
            emotion_counts.get(emotion, 0) + 1
        )

    dominant_emotion = max(
        emotion_counts,
        key=emotion_counts.get
    )


    average_intensity = int(
        np.mean([
            r["intensity"]
            for r in results
        ])
    )


    if len(results) > 1:

        switch_ratio = int(
            switch_count /
            (len(results) - 1) *
            100
        )

    else:

        switch_ratio = 0

    if switch_ratio >= 75:
        stability = "🌪️ Highly Unstable"
    elif switch_ratio >= 50:
        stability = "⚡ Frequently Changing"
    elif switch_ratio >= 25:
        stability = "🌤️ Moderately Stable"
    else:
        stability = "🧘 Very Stable"


    if switch_count == 0:

        insight = (
            "🧘 One emotion throughout. "
            "This conversation is surprisingly peaceful."
        )

    elif switch_count == 1:

        insight = (
            "👀 One emotional plot twist detected."
        )

    elif switch_count <= 3:

        insight = (
            "🎭 Some emotional plot twists are happening."
        )

    elif switch_count <= 5:

        insight = (
            "🔥 This conversation has serious character development."
        )

    else:

        insight = (
            "💀 Emotional roller coaster detected. "
            "Someone needs a seatbelt."
        )


    table_data = []

    for result in results:

        table_data.append([
            result["sentence_no"],
            result["sentence"],
            result["emotion"],
            result["intensity"],
            result["level"]
        ])

    table = pd.DataFrame(
        table_data,
        columns=[
            "Sentence",
            "Text",
            "Emotion",
            "Intensity",
            "Level"
        ]
    )

    analysis_history.append({
        "Time": datetime.now().strftime("%H:%M:%S"),
        "Sentences": len(sentences),
        "Switches": switch_count,
        "Dominant Emotion": dominant_emotion,
        "Avg Intensity": average_intensity
    })



    emotion_names = [
        r["emotion"]
        for r in results
    ]

    emotion_numbers = []

    emotion_index = {
        "😐 Neutral": 0,
        "😊 Happy": 1,
        "😢 Sad": 2,
        "😡 Angry": 3,
        "😨 Fear": 4,
        "😲 Surprise": 5,
        "😍 Love": 6
    }

    for emotion in emotion_names:

        emotion_numbers.append(
            emotion_index.get(
                emotion,
                0
            )
        )

    fig = plt.figure(
        figsize=(10, 4)
    )

    plt.plot(
        range(
            1,
            len(emotion_numbers) + 1
        ),
        emotion_numbers,
        marker="o",
        linewidth=2
    )

    plt.yticks(
        list(emotion_index.values()),
        list(emotion_index.keys())
    )

    plt.xlabel("Sentence Number")
    plt.ylabel("Emotion")
    plt.title("Emotion Timeline")

    plt.grid(
        alpha=0.2
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return (
        f"{switch_count}",
        dominant_emotion,
        stability,
        f"{average_intensity}/100",
        insight,
        table,
        fig
    )



def analyze_audio(audio):

    if audio is None:

        return (
            "—",
            "—",
            "—",
            "Please record or upload audio first."
        )

    try:

        y, sr = librosa.load(
            audio,
            sr=None
        )

        if len(y) == 0:
            raise Exception()

        duration = len(y) / sr


        rms = librosa.feature.rms(
            y=y
        )[0]

        energy = float(
            np.mean(rms)
        )


        zcr = librosa.feature.zero_crossing_rate(
            y
        )[0]

        zcr_value = float(
            np.mean(zcr)
        )



        energy_score = min(
            100,
            int(energy * 500)
        )



        intervals = librosa.effects.split(
            y,
            top_db=30
        )

        voiced_duration = sum(
            (end - start) / sr
            for start, end in intervals
        )

        pause_ratio = max(
            0,
            1 - voiced_duration / max(
                duration,
                0.1
            )
        )

        pause_percentage = int(
            pause_ratio * 100
        )


        if energy_score >= 70:
            voice_emotion = "🔥 High Emotional Energy"

        elif energy_score >= 40:
            voice_emotion = "🙂 Moderate Energy"

        else:
            voice_emotion = "😴 Low Energy"

        if pause_percentage > 50:

            insight = (
                "Long pauses detected. "
                "The speaker sounds hesitant or reflective."
            )

        elif energy_score > 70:

            insight = (
                "High energy detected. "
                "The speaker sounds emotionally intense."
            )

        else:

            insight = (
                "Speech energy appears relatively balanced."
            )

        return (
            f"{duration:.2f} sec",
            f"{energy_score}/100",
            f"{pause_percentage}%",
            f"{voice_emotion} — {insight}"
        )

    except:

        return (
            "—",
            "—",
            "—",
            "Unable to analyze this audio."
        )



def sample_text():

    return """
I was so happy when I got selected for the project.
Everyone congratulated me and I was really excited!
But then my teammate said the project was cancelled.
I was shocked and couldn't believe it.
Why does this always happen to me?!
After some time, our professor gave us another chance.
I felt relieved and happy again.
"""


def sample_calm():

    return """
Today I went to college.
I attended my classes.
After lunch I worked on my project.
In the evening I returned home.
"""


def sample_rollercoaster():

    return """
I was extremely excited about the competition!
Everything was going perfectly.
Suddenly I realized that I had forgotten my presentation.
I was terrified.
Then my friend helped me.
I felt so relieved and happy.
We finally won the competition!
"""


def show_history():

    if not analysis_history:

        return pd.DataFrame(
            columns=[
                "Time",
                "Sentences",
                "Switches",
                "Dominant Emotion",
                "Avg Intensity"
            ]
        )

    return pd.DataFrame(
        analysis_history
    )


def clear_history():

    analysis_history.clear()

    return pd.DataFrame(
        columns=[
            "Time",
            "Sentences",
            "Switches",
            "Dominant Emotion",
            "Avg Intensity"
        ]
    )

css = """

body {
    background: #08090d !important;
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    background: #08090d !important;
}

#hero {
    text-align: center;
    padding: 35px;
    border-radius: 25px;
    background: linear-gradient(
        135deg,
        #101827,
        #1e1635,
        #351329
    );
    border: 1px solid #3c3150;
    margin-bottom: 20px;
}

#hero h1 {
    font-size: 46px !important;
    font-weight: 900 !important;
}

#hero p {
    color: #b8b8c5 !important;
    font-size: 17px !important;
}

.card {
    background: #11131a !important;
    border: 1px solid #292c38 !important;
    border-radius: 20px !important;
    padding: 20px !important;
}

.big-number {
    font-size: 30px !important;
    font-weight: 800 !important;
}

button {
    border-radius: 13px !important;
}

textarea {
    border-radius: 14px !important;
}

"""


with gr.Blocks(
    title="🧠 Emotion Switch Detector",
    theme=gr.themes.Base(
        primary_hue="purple",
        neutral_hue="slate"
    ),
    css=css
) as app:

    gr.HTML("""
    <div id="hero">

        <h1>🧠 EMOTION SWITCH DETECTOR</h1>

        <p>
        Discover how emotions change throughout a conversation.
        </p>

        <p>
        <b>One sentence at a time → One emotional journey.</b>
        </p>

    </div>
    """)

    gr.Markdown("## 💬 Analyze a Conversation")

    with gr.Row():

        with gr.Column(
            scale=2,
            elem_classes="card"
        ):

            text_input = gr.Textbox(
                label="Conversation / Story",
                placeholder=
                "Enter a story, conversation, message or paragraph...",
                lines=10
            )

            with gr.Row():

                analyze_btn = gr.Button(
                    "🧠 ANALYZE EMOTIONS",
                    variant="primary"
                )

                example_btn = gr.Button(
                    "🎭 EMOTIONAL STORY"
                )

        with gr.Column(
            scale=1,
            elem_classes="card"
        ):

            gr.Markdown("""
            ### 🔍 What will be detected?

            😊 Happy
            😢 Sad
            😡 Angry
            😨 Fear
            😲 Surprise
            😍 Love
            😐 Neutral

            The system checks each sentence
            and detects emotional transitions.
            """)

    # --------------------------------------------------------
    # DASHBOARD
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## 📊 Emotion Dashboard")

    with gr.Row():

        switch_count = gr.Textbox(
            label="🔄 Emotion Switches",
            elem_classes="big-number"
        )

        dominant = gr.Textbox(
            label="🎯 Dominant Emotion"
        )

        stability = gr.Textbox(
            label="🌡️ Emotional Stability"
        )

        intensity = gr.Textbox(
            label="⚡ Average Intensity"
        )

    insight = gr.Textbox(
        label="🤖 AI Insight",
        lines=2
    )

    # --------------------------------------------------------
    # TABLE
    # --------------------------------------------------------

    gr.Markdown("## 🔬 Sentence-by-Sentence Analysis")

    result_table = gr.Dataframe(
        headers=[
            "Sentence",
            "Text",
            "Emotion",
            "Intensity",
            "Level"
        ],
        interactive=False,
        wrap=True
    )

    # --------------------------------------------------------
    # GRAPH
    # --------------------------------------------------------

    gr.Markdown("## 📈 Emotion Timeline")

    emotion_graph = gr.Plot(
        label="Emotion Changes"
    )

    # --------------------------------------------------------
    # QUICK EXAMPLES
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## ⚡ Quick Test Cases")

    with gr.Row():

        calm_btn = gr.Button(
            "🧘 Calm Story"
        )

        roller_btn = gr.Button(
            "🎢 Emotional Rollercoaster"
        )

    # --------------------------------------------------------
    # AUDIO
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## 🎤 Speech Emotion Analysis")

    with gr.Row():

        with gr.Column(
            elem_classes="card"
        ):

            audio_input = gr.Audio(
                sources=[
                    "microphone",
                    "upload"
                ],
                type="filepath",
                label="🎤 Record / Upload Speech"
            )

            audio_btn = gr.Button(
                "🎤 ANALYZE SPEECH",
                variant="primary"
            )

        with gr.Column(
            elem_classes="card"
        ):

            audio_duration = gr.Textbox(
                label="⏱️ Duration"
            )

            audio_energy = gr.Textbox(
                label="⚡ Voice Energy"
            )

            audio_pause = gr.Textbox(
                label="⏸️ Pause Ratio"
            )

    audio_result = gr.Textbox(
        label="🎙️ Speech Analysis",
        lines=3
    )

    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## 📜 Analysis History")

    history_table = gr.Dataframe(
        headers=[
            "Time",
            "Sentences",
            "Switches",
            "Dominant Emotion",
            "Avg Intensity"
        ],
        interactive=False
    )

    with gr.Row():

        history_btn = gr.Button(
            "📊 SHOW HISTORY"
        )

        clear_btn = gr.Button(
            "🗑️ CLEAR HISTORY"
        )

    gr.HTML("""
    <div style="
        text-align:center;
        padding:30px;
        color:#666;
    ">
        🧠 Emotion Switch Detector
        <br>
        <small>
        Text & Speech Analysis Mini Project
        </small>
        <br><br>
        <small>
        "Because emotions don't know how to stay in one lane." 🎭
        </small>
    </div>
    """)

    analyze_btn.click(
        fn=analyze_text,
        inputs=text_input,
        outputs=[
            switch_count,
            dominant,
            stability,
            intensity,
            insight,
            result_table,
            emotion_graph
        ]
    )

    example_btn.click(
        fn=sample_text,
        inputs=None,
        outputs=text_input
    )

    calm_btn.click(
        fn=sample_calm,
        inputs=None,
        outputs=text_input
    )

    roller_btn.click(
        fn=sample_rollercoaster,
        inputs=None,
        outputs=text_input
    )

    audio_btn.click(
        fn=analyze_audio,
        inputs=audio_input,
        outputs=[
            audio_duration,
            audio_energy,
            audio_pause,
            audio_result
        ]
    )

    history_btn.click(
        fn=show_history,
        inputs=None,
        outputs=history_table
    )

    clear_btn.click(
        fn=clear_history,
        inputs=None,
        outputs=history_table
    )


app.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_2578/685095272.py:804: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84b53c913e34eaf834.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
